# Esports Market Map And Latency-Arb Visuals

Companion notebook for `notes/esports_latency_arb_market_map.md`.

This notebook uses two data layers:

- Current Gamma event pulls for `series_id`, event slugs, sub-market slugs, `sportsMarketType`, and tournament labels.
- Local parquet for historical fills and closed-position behaviour: `data/trades/*.parquet`, `data/markets/markets_*.parquet`, and `data/closed_positions.parquet`.

Charts are rendered as inline SVG so the notebook does not require matplotlib, plotly, or seaborn.

In [ ]:

from __future__ import annotations

import html
import time
from pathlib import Path

import duckdb
import httpx
import pandas as pd
from IPython.display import SVG, display

ROOT = Path.cwd().resolve()
while not (ROOT / "polymarket" / "research" / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RESEARCH = ROOT / "polymarket" / "research"
DATA = RESEARCH / "data"
CACHE = DATA / "analysis" / "esports_market_map"
CACHE.mkdir(parents=True, exist_ok=True)

MARKETS_DIR = DATA / "markets"
MARKETS_PATH = sorted(MARKETS_DIR.glob("markets_*.parquet"))[-1]
TRADES_GLOB = str(DATA / "trades" / "trades_delta_shard*.parquet")
SEED_PATH = DATA / "trades" / "trades_seed.parquet"
CLOSED_POSITIONS_PATH = DATA / "closed_positions.parquet"

ANALYSIS_AS_OF = pd.Timestamp("2026-05-19T00:00:00Z")
GAMMA_BASE = "https://gamma-api.polymarket.com"
SERIES = {
    "lol": ("10311", "League of Legends"),
    "cs2": ("10310", "Counter Strike 2"),
    "dota2": ("10309", "Dota 2"),
    "valorant": ("10369", "Valorant"),
    "mlbb": ("10426", "Mobile Legends: Bang Bang"),
    "cod": ("10427", "Call of Duty"),
    "wildrift": ("10429", "League of Legends: Wild Rift"),
    "overwatch": ("10430", "Overwatch"),
    "pubg": ("10431", "PUBG"),
    "r6": ("10432", "Rainbow Six Siege"),
    "rocketleague": ("10433", "Rocket League"),
    "hok": ("10434", "Honor of Kings"),
    "sc2": ("10435", "StarCraft 2"),
    "scbw": ("10436", "StarCraft: Brood War"),
}

con = duckdb.connect()
con.execute("PRAGMA threads=8")
print(f"repo root     : {ROOT}")
print(f"markets       : {MARKETS_PATH.name}")
print(f"cache dir     : {CACHE}")
print(f"analysis as of: {ANALYSIS_AS_OF}")


In [ ]:

GAME_CASE = """
CASE
  WHEN regexp_matches(slug, '^lol-') OR lower(question) LIKE '%league of legends%' OR lower(question) LIKE 'lol:%' THEN 'League of Legends'
  WHEN regexp_matches(slug, '^cs2-') OR lower(question) LIKE '%counter-strike%' OR lower(question) LIKE '%cs2:%' THEN 'Counter Strike 2'
  WHEN regexp_matches(slug, '^dota2-') OR lower(question) LIKE '%dota 2%' THEN 'Dota 2'
  WHEN regexp_matches(slug, '^val-') OR lower(question) LIKE '%valorant%' THEN 'Valorant'
  WHEN regexp_matches(slug, '^mlbb-') OR lower(question) LIKE '%mobile legends%' THEN 'Mobile Legends: Bang Bang'
  WHEN regexp_matches(slug, '^cod-') OR lower(question) LIKE '%call of duty%' THEN 'Call of Duty'
  WHEN regexp_matches(slug, '^ow-') OR lower(question) LIKE '%overwatch%' THEN 'Overwatch'
  WHEN regexp_matches(slug, '^r6siege-') OR lower(question) LIKE '%rainbow six%' THEN 'Rainbow Six Siege'
  WHEN regexp_matches(slug, '^rl-') OR lower(question) LIKE '%rocket league%' THEN 'Rocket League'
  WHEN regexp_matches(slug, '^hok-') OR lower(question) LIKE '%honor of kings%' THEN 'Honor of Kings'
  WHEN regexp_matches(slug, '^sc2-') OR lower(question) LIKE '%starcraft 2%' THEN 'StarCraft 2'
  WHEN regexp_matches(slug, '^scbw-') OR lower(question) LIKE '%brood war%' THEN 'StarCraft: Brood War'
  WHEN regexp_matches(slug, '^pubg-') OR lower(question) LIKE '%pubg%' THEN 'PUBG'
  WHEN regexp_matches(slug, '^wr-') OR lower(question) LIKE '%wild rift%' THEN 'League of Legends: Wild Rift'
END
"""

MARKET_TYPE_CASE = """
CASE
  WHEN regexp_matches(slug, 'game[0-9]+-kill-over|game[0-9]+-kill-under|kill-over|kill-under') THEN 'live_total_kills'
  WHEN regexp_matches(slug, 'odd-even-total-kills') THEN 'odd_even_total_kills'
  WHEN regexp_matches(slug, 'odd-even-total-rounds') THEN 'odd_even_total_rounds'
  WHEN regexp_matches(slug, 'game[0-9]+$') THEN 'game_winner'
  WHEN regexp_matches(slug, 'total-games') THEN 'series_total_games'
  WHEN regexp_matches(slug, 'handicap') THEN 'handicap'
  WHEN regexp_matches(slug, 'first-blood') THEN 'first_blood'
  WHEN regexp_matches(slug, 'baron|dragon|roshan|barracks|inhibitor|tower|nashor|quadra|penta|rampage|ultra') THEN 'objective_or_multikill_prop'
  ELSE 'series_moneyline_or_other'
END
"""

ESPORTS_MARKETS_CTE = f"""
esports_markets AS (
    SELECT
        CAST(id AS VARCHAR) AS market_id,
        condition_id,
        slug,
        question,
        active,
        closed,
        TRY_CAST(end_date AS TIMESTAMP) AS end_date,
        volume,
        liquidity,
        {GAME_CASE} AS game,
        {MARKET_TYPE_CASE} AS market_type_proxy
    FROM read_parquet('{MARKETS_PATH}')
)
"""

RAW_TRADES_CTE = f"""
raw_trades AS (
    SELECT * FROM read_parquet('{TRADES_GLOB}')
    UNION ALL BY NAME
    SELECT * FROM read_parquet('{SEED_PATH}')
)
"""


In [ ]:

PALETTE = ["#145C9E", "#D95F02", "#1B9E77", "#7570B3", "#E7298A", "#66A61E", "#E6AB02", "#A6761D", "#666666"]

def _fmt_num(x):
    if pd.isna(x):
        return ""
    x = float(x)
    if abs(x) >= 1e9:
        return f"{x/1e9:.2f}B"
    if abs(x) >= 1e6:
        return f"{x/1e6:.1f}M"
    if abs(x) >= 1e3:
        return f"{x/1e3:.0f}K"
    return f"{x:.0f}"


def bar_chart(df, label_col, value_col, title, width=920, row_h=30, color="#145C9E", value_fmt=_fmt_num):
    d = df[[label_col, value_col]].dropna().copy().sort_values(value_col, ascending=True)
    max_v = float(d[value_col].max()) if len(d) else 1.0
    left, right, top = 230, 130, 44
    height = top + max(1, len(d)) * row_h + 20
    plot_w = width - left - right
    parts = [f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>"]
    parts.append("<style>text{font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;font-size:13px;fill:#222}.title{font-weight:650;font-size:18px}</style>")
    parts.append(f"<text class='title' x='0' y='22'>{html.escape(title)}</text>")
    for i, row in enumerate(d.itertuples(index=False)):
        label = str(getattr(row, label_col)); v = float(getattr(row, value_col) or 0)
        y = top + i * row_h; w = 0 if max_v <= 0 else plot_w * v / max_v
        parts.append(f"<text x='{left-8}' y='{y+18}' text-anchor='end'>{html.escape(label)}</text>")
        parts.append(f"<rect x='{left}' y='{y+5}' width='{w:.1f}' height='18' rx='3' fill='{color}' opacity='0.88'/>")
        parts.append(f"<text x='{left+w+6}' y='{y+19}'>{html.escape(value_fmt(v))}</text>")
    parts.append("</svg>")
    display(SVG("".join(parts)))


def grouped_bar_chart(df, group_col, value_cols, title, width=940, row_h=34):
    d = df[[group_col] + value_cols].copy().sort_values(value_cols[0], ascending=True)
    max_v = max(float(d[c].max() or 0) for c in value_cols) or 1.0
    left, right, top = 230, 120, 52
    plot_w = width - left - right
    height = top + len(d) * row_h + 45
    parts = [f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>"]
    parts.append("<style>text{font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;font-size:13px;fill:#222}.title{font-weight:650;font-size:18px}.legend{font-size:12px;fill:#555}</style>")
    parts.append(f"<text class='title' x='0' y='22'>{html.escape(title)}</text>")
    for j, col in enumerate(value_cols):
        x = left + j * 170
        parts.append(f"<rect x='{x}' y='34' width='12' height='12' fill='{PALETTE[j]}' rx='2'/><text class='legend' x='{x+18}' y='45'>{html.escape(col)}</text>")
    for i, row in enumerate(d.itertuples(index=False)):
        label = str(getattr(row, group_col)); y = top + i * row_h
        parts.append(f"<text x='{left-8}' y='{y+20}' text-anchor='end'>{html.escape(label)}</text>")
        for j, col in enumerate(value_cols):
            v = float(getattr(row, col) or 0); w = plot_w * v / max_v; yy = y + 4 + j * 12
            parts.append(f"<rect x='{left}' y='{yy}' width='{w:.1f}' height='10' fill='{PALETTE[j]}' opacity='0.86'/>")
            if j == 0:
                parts.append(f"<text x='{left+w+6}' y='{yy+10}'>{_fmt_num(v)}</text>")
    parts.append("</svg>")
    display(SVG("".join(parts)))


def line_chart(df, x_col, y_col, series_col, title, width=940, height=430, y_fmt=_fmt_num, max_series=6):
    d = df[[x_col, y_col, series_col]].dropna().copy()
    order = d.groupby(series_col)[y_col].sum().sort_values(ascending=False).head(max_series).index.tolist()
    d = d[d[series_col].isin(order)]
    if d.empty:
        print("No data to chart"); return
    d[x_col] = pd.to_datetime(d[x_col])
    xs = sorted(d[x_col].unique())
    x_index = {x: i for i, x in enumerate(xs)}
    min_y, max_y = min(0, float(d[y_col].min())), float(d[y_col].max()) or 1.0
    left, right, top, bottom = 72, 170, 48, 52
    plot_w, plot_h = width-left-right, height-top-bottom
    def px(x): return left + (plot_w * x_index[x] / max(1, len(xs)-1))
    def py(y): return top + plot_h - (plot_h * (float(y)-min_y) / max(1e-9, max_y-min_y))
    parts = [f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>"]
    parts.append("<style>text{font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;font-size:12px;fill:#222}.title{font-weight:650;font-size:18px}.grid{stroke:#e6e6e6;stroke-width:1}.axis{stroke:#888;stroke-width:1}.legend{font-size:12px}</style>")
    parts.append(f"<text class='title' x='0' y='22'>{html.escape(title)}</text>")
    for frac in [0, .25, .5, .75, 1]:
        y = top + plot_h * frac; val = max_y - (max_y-min_y) * frac
        parts.append(f"<line class='grid' x1='{left}' x2='{left+plot_w}' y1='{y:.1f}' y2='{y:.1f}'/>")
        parts.append(f"<text x='{left-8}' y='{y+4:.1f}' text-anchor='end'>{html.escape(y_fmt(val))}</text>")
    parts.append(f"<line class='axis' x1='{left}' x2='{left+plot_w}' y1='{top+plot_h}' y2='{top+plot_h}'/>")
    for idx, (name, g) in enumerate(d.groupby(series_col, sort=False)):
        color = PALETTE[idx % len(PALETTE)]; g = g.sort_values(x_col)
        pts = " ".join(f"{px(r[x_col]):.1f},{py(r[y_col]):.1f}" for _, r in g.iterrows())
        parts.append(f"<polyline points='{pts}' fill='none' stroke='{color}' stroke-width='2.2'/>")
        lx, ly = width-right+20, top + idx*22
        parts.append(f"<rect x='{lx}' y='{ly-10}' width='12' height='12' fill='{color}' rx='2'/><text class='legend' x='{lx+18}' y='{ly}'>{html.escape(str(name))}</text>")
    for x in xs[::max(1, len(xs)//6)]:
        parts.append(f"<text x='{px(x):.1f}' y='{height-18}' text-anchor='middle'>{pd.Timestamp(x).strftime('%Y-%m')}</text>")
    parts.append("</svg>")
    display(SVG("".join(parts)))


def heatmap(df, row_col, col_col, value_col, title, width=980, cell_w=96, cell_h=28):
    pivot = df.pivot_table(index=row_col, columns=col_col, values=value_col, aggfunc='sum', fill_value=0)
    if pivot.empty:
        print("No data to chart"); return
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
    cols, rows = list(pivot.columns), list(pivot.index)
    left, top = 230, 74
    height = top + len(rows) * cell_h + 22
    width = max(width, left + len(cols) * cell_w + 30)
    max_v = float(pivot.to_numpy().max()) or 1.0
    parts = [f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>"]
    parts.append("<style>text{font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;font-size:12px;fill:#222}.title{font-weight:650;font-size:18px}.col{font-size:11px;fill:#444}</style>")
    parts.append(f"<text class='title' x='0' y='22'>{html.escape(title)}</text>")
    for j, col in enumerate(cols):
        cx = left+j*cell_w+cell_w/2
        parts.append(f"<text class='col' x='{cx}' y='58' text-anchor='middle' transform='rotate(-25 {cx} 58)'>{html.escape(str(col))}</text>")
    for i, row in enumerate(rows):
        y = top + i * cell_h
        parts.append(f"<text x='{left-8}' y='{y+18}' text-anchor='end'>{html.escape(str(row))}</text>")
        for j, col in enumerate(cols):
            v = float(pivot.loc[row, col]); alpha = 0.08 + 0.82 * (v / max_v); x = left + j * cell_w
            parts.append(f"<rect x='{x}' y='{y}' width='{cell_w-3}' height='{cell_h-3}' fill='#145C9E' opacity='{alpha:.3f}' rx='3'/>")
            if v > 0:
                parts.append(f"<text x='{x+cell_w/2}' y='{y+18}' text-anchor='middle' fill='#111'>{html.escape(_fmt_num(v))}</text>")
    parts.append("</svg>")
    display(SVG("".join(parts)))


## 1. Current Gamma Coverage

This pulls active, unclosed events by esports series. `open_*` includes stale unresolved markets; `near_*` keeps only markets with `end_date >= ANALYSIS_AS_OF`.

In [ ]:

def _tournament(ev: dict) -> str:
    title = ev.get("title") or ""
    if " - " in title:
        return title.split(" - ", 1)[1].strip()
    desc = ev.get("description") or ""
    marker = " in the "
    if marker in desc:
        return desc.split(marker, 1)[1].split(".", 1)[0].split("\n", 1)[0].strip()
    return "TBD/unspecified"


def fetch_gamma_current(use_cache=True):
    cache_path = CACHE / f"gamma_esports_current_{ANALYSIS_AS_OF.strftime('%Y%m%d')}.parquet"
    if use_cache and cache_path.exists():
        return pd.read_parquet(cache_path)
    rows = []
    with httpx.Client(base_url=GAMMA_BASE, timeout=30.0, headers={"User-Agent": "Mozilla/5.0"}) as client:
        for game_key, (sid, game) in SERIES.items():
            offset = 0
            while True:
                r = client.get("/events", params={
                    "series_id": sid, "active": "true", "closed": "false",
                    "archived": "false", "limit": 100, "offset": offset,
                })
                r.raise_for_status()
                events = r.json()
                if not events:
                    break
                for ev in events:
                    for m in ev.get("markets") or []:
                        rows.append({
                            "game_key": game_key, "game": game, "series_id": sid,
                            "event_slug": ev.get("slug"), "event_title": ev.get("title"),
                            "tournament": _tournament(ev), "market_slug": m.get("slug"),
                            "sportsMarketType": m.get("sportsMarketType") or "unknown",
                            "line": m.get("line"), "condition_id": m.get("conditionId"),
                            "question": m.get("question"),
                            "volume": float(m.get("volumeNum") or m.get("volume") or 0),
                            "end_date": m.get("endDate") or ev.get("endDate"),
                        })
                if len(events) < 100:
                    break
                offset += 100
                time.sleep(0.05)
    markets = pd.DataFrame(rows)
    if not markets.empty:
        markets["end_date"] = pd.to_datetime(markets["end_date"], utc=True, errors="coerce")
        markets.to_parquet(cache_path, index=False)
    return markets

current = fetch_gamma_current(use_cache=True)
near = current[current["end_date"] >= ANALYSIS_AS_OF].copy()
coverage = (current.groupby(["game", "series_id"])
    .agg(open_events=("event_slug", "nunique"), open_markets=("market_slug", "count"), open_volume=("volume", "sum"))
    .reset_index())
near_cov = (near.groupby("game")
    .agg(near_events=("event_slug", "nunique"), near_markets=("market_slug", "count"), near_volume=("volume", "sum"))
    .reset_index())
coverage = coverage.merge(near_cov, on="game", how="left").fillna({"near_events":0, "near_markets":0, "near_volume":0})
coverage = coverage.sort_values("near_markets", ascending=False)
display(coverage)
grouped_bar_chart(coverage, "game", ["near_markets", "open_markets"], "Current Gamma esports sub-markets: near-term vs all open")


In [ ]:

type_near = near.groupby(["game", "sportsMarketType"]).size().reset_index(name="markets")
heatmap(type_near, "game", "sportsMarketType", "markets", "Near-term Gamma market types by game")

top_tournaments = (near.groupby(["game", "tournament"])
    .agg(markets=("market_slug", "count"), events=("event_slug", "nunique"), volume=("volume", "sum"))
    .sort_values("markets", ascending=False)
    .reset_index()
    .head(30))
display(top_tournaments)


## 2. Local Market Snapshot

This is the historical market universe in `data/markets/markets_*.parquet`, classified by slug/question. It is separate from the current Gamma event pull because the local market snapshot lacks `series_id` and `sportsMarketType`.

In [ ]:

local_markets = con.sql(f"""
WITH {ESPORTS_MARKETS_CTE}
SELECT game,
       count(*) AS markets,
       sum(CASE WHEN closed THEN 1 ELSE 0 END) AS closed_markets,
       sum(CASE WHEN active THEN 1 ELSE 0 END) AS active_flag_markets,
       sum(coalesce(volume, 0)) AS gamma_volume,
       sum(coalesce(liquidity, 0)) AS gamma_liquidity,
       min(end_date) AS min_end_date,
       max(end_date) AS max_end_date
FROM esports_markets
WHERE game IS NOT NULL
GROUP BY game
ORDER BY gamma_volume DESC
""").fetchdf()
display(local_markets)
bar_chart(local_markets, "game", "markets", "Local Gamma snapshot: esports markets by game", color="#1B9E77")


## 3. Historical Raw Fill Aggregates

This scans the local `OrderFilled` parquet layer, including closed trades. The first run writes a compact cache in `data/analysis/esports_market_map/`; later runs load that cache.

In [ ]:

def historical_fill_aggregates(use_cache=True):
    cache_path = CACHE / "historical_fill_aggregates.parquet"
    if use_cache and cache_path.exists():
        return pd.read_parquet(cache_path)
    sql = f"""
    COPY (
    WITH {ESPORTS_MARKETS_CTE}, {RAW_TRADES_CTE}, joined AS (
        SELECT date_trunc('month', t.timestamp) AS month,
               em.game, em.market_type_proxy, t.market_id,
               t.usd_amount, t.price, t.maker_side
        FROM raw_trades t
        JOIN esports_markets em ON em.market_id = t.market_id
        WHERE em.game IS NOT NULL
    )
    SELECT 'by_game' AS table_name, game, NULL::TIMESTAMP AS month, NULL::VARCHAR AS market_type_proxy,
           count(*) AS fills, count(DISTINCT market_id) AS markets_traded,
           sum(usd_amount) AS usd_notional, avg(price) AS avg_price
    FROM joined GROUP BY game
    UNION ALL
    SELECT 'by_game_month' AS table_name, game, month, NULL::VARCHAR AS market_type_proxy,
           count(*) AS fills, count(DISTINCT market_id) AS markets_traded,
           sum(usd_amount) AS usd_notional, avg(price) AS avg_price
    FROM joined GROUP BY game, month
    UNION ALL
    SELECT 'by_game_type' AS table_name, game, NULL::TIMESTAMP AS month, market_type_proxy,
           count(*) AS fills, count(DISTINCT market_id) AS markets_traded,
           sum(usd_amount) AS usd_notional, avg(price) AS avg_price
    FROM joined GROUP BY game, market_type_proxy
    ) TO '{cache_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    t0 = time.time(); con.execute(sql); print(f"built {cache_path.name} in {time.time() - t0:.1f}s")
    return pd.read_parquet(cache_path)

hist = historical_fill_aggregates(use_cache=True)
by_game = hist[hist.table_name == "by_game"].sort_values("usd_notional", ascending=False)
display(by_game)
bar_chart(by_game, "game", "usd_notional", "Historical esports raw notional by game", color="#D95F02")


In [ ]:

monthly = hist[hist.table_name == "by_game_month"].copy()
monthly["month"] = pd.to_datetime(monthly["month"])
line_chart(monthly, "month", "usd_notional", "game", "Monthly esports raw notional by game", max_series=5)

type_hist = hist[hist.table_name == "by_game_type"].copy()
heatmap(type_hist, "game", "market_type_proxy", "usd_notional", "Historical notional by game and market-type proxy")
display(type_hist.sort_values("usd_notional", ascending=False).head(30))


## 4. Fill Cadence Proxy

For each outcome token, compute gaps between consecutive fills, then summarise by game. This is a market-activity proxy, not direct source latency. Same-timestamp fills and same-block clusters produce 0 second gaps.

In [ ]:

def latency_proxy(use_cache=True):
    cache_path = CACHE / "latency_proxy_by_game.parquet"
    if use_cache and cache_path.exists():
        return pd.read_parquet(cache_path)
    sql = f"""
    COPY (
    WITH {ESPORTS_MARKETS_CTE}, raw_trades AS (
        SELECT timestamp, market_id, maker_asset_id, taker_asset_id, usd_amount FROM read_parquet('{TRADES_GLOB}')
        UNION ALL BY NAME
        SELECT timestamp, market_id, maker_asset_id, taker_asset_id, usd_amount FROM read_parquet('{SEED_PATH}')
    ), esports_fills AS (
        SELECT em.game, t.market_id,
               CASE WHEN maker_asset_id='0' THEN taker_asset_id ELSE maker_asset_id END AS outcome_token_id,
               timestamp, usd_amount
        FROM raw_trades t
        JOIN esports_markets em USING (market_id)
        WHERE em.game IS NOT NULL
    ), gaps AS (
        SELECT game, market_id, outcome_token_id, usd_amount,
               epoch(timestamp - lag(timestamp) OVER (PARTITION BY outcome_token_id ORDER BY timestamp)) AS gap_seconds
        FROM esports_fills
    )
    SELECT game,
           count(*) FILTER (WHERE gap_seconds IS NOT NULL) AS observed_gaps,
           quantile_cont(gap_seconds, 0.10) FILTER (WHERE gap_seconds BETWEEN 0 AND 86400) AS gap_p10_s,
           quantile_cont(gap_seconds, 0.25) FILTER (WHERE gap_seconds BETWEEN 0 AND 86400) AS gap_p25_s,
           quantile_cont(gap_seconds, 0.50) FILTER (WHERE gap_seconds BETWEEN 0 AND 86400) AS gap_p50_s,
           quantile_cont(gap_seconds, 0.75) FILTER (WHERE gap_seconds BETWEEN 0 AND 86400) AS gap_p75_s,
           quantile_cont(gap_seconds, 0.90) FILTER (WHERE gap_seconds BETWEEN 0 AND 86400) AS gap_p90_s,
           avg(CASE WHEN gap_seconds <= 5 THEN 1.0 ELSE 0.0 END) FILTER (WHERE gap_seconds IS NOT NULL AND gap_seconds BETWEEN 0 AND 86400) AS pct_gap_le_5s,
           avg(CASE WHEN gap_seconds <= 30 THEN 1.0 ELSE 0.0 END) FILTER (WHERE gap_seconds IS NOT NULL AND gap_seconds BETWEEN 0 AND 86400) AS pct_gap_le_30s,
           avg(CASE WHEN gap_seconds <= 300 THEN 1.0 ELSE 0.0 END) FILTER (WHERE gap_seconds IS NOT NULL AND gap_seconds BETWEEN 0 AND 86400) AS pct_gap_le_5m
    FROM gaps
    GROUP BY game
    ORDER BY observed_gaps DESC
    ) TO '{cache_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    t0 = time.time(); con.execute(sql); print(f"built {cache_path.name} in {time.time() - t0:.1f}s")
    return pd.read_parquet(cache_path)

lat = latency_proxy(use_cache=True)
display(lat)
grouped_bar_chart(lat.sort_values("pct_gap_le_30s", ascending=False), "game", ["pct_gap_le_5s", "pct_gap_le_30s", "pct_gap_le_5m"], "Share of historical same-outcome fills inside latency windows")


## 5. Top Historical Esports Markets

Useful for selecting case studies where liquidity existed and the order book likely had enough activity to test stale-price behaviour.

In [ ]:

def top_historical_markets(use_cache=True):
    cache_path = CACHE / "top_historical_markets.parquet"
    if use_cache and cache_path.exists():
        return pd.read_parquet(cache_path)
    sql = f"""
    COPY (
    WITH {ESPORTS_MARKETS_CTE}, {RAW_TRADES_CTE}
    SELECT em.game, em.market_type_proxy, em.market_id,
           any_value(em.slug) AS market_slug,
           any_value(em.question) AS question,
           count(*) AS fills,
           sum(t.usd_amount) AS usd_notional,
           min(t.timestamp) AS first_fill,
           max(t.timestamp) AS last_fill
    FROM raw_trades t
    JOIN esports_markets em ON em.market_id = t.market_id
    WHERE em.game IS NOT NULL
    GROUP BY em.game, em.market_type_proxy, em.market_id
    ORDER BY usd_notional DESC
    LIMIT 250
    ) TO '{cache_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    t0 = time.time(); con.execute(sql); print(f"built {cache_path.name} in {time.time() - t0:.1f}s")
    return pd.read_parquet(cache_path)

top_markets = top_historical_markets(use_cache=True)
display(top_markets.head(30))
bar_chart(top_markets.head(20), "market_slug", "usd_notional", "Top historical esports markets by raw notional", width=1120, color="#7570B3")


## 6. Closed-Position Summary

This uses `closed_positions.parquet`, so it is on resolved markets only. Aggregate PnL should net to roughly zero because Polymarket is a closed system; the useful fields here are position counts, held-to-resolution rate, and gross exposure by game.

In [ ]:

def closed_position_summary(use_cache=True):
    cache_path = CACHE / "closed_position_summary.parquet"
    if use_cache and cache_path.exists():
        return pd.read_parquet(cache_path)
    sql = f"""
    COPY (
    WITH {ESPORTS_MARKETS_CTE}
    SELECT em.game,
           count(*) AS positions,
           count(DISTINCT cp.market_id) AS markets,
           count(DISTINCT address) AS addresses,
           sum(gross_usd_volume) AS gross_usd_volume,
           sum(realised_pnl) AS realised_pnl,
           avg(CASE WHEN is_held_to_resolution THEN 1.0 ELSE 0.0 END) AS held_to_resolution_rate,
           avg(CASE WHEN realised_pnl > 0 THEN 1.0 ELSE 0.0 END) AS win_position_rate
    FROM read_parquet('{CLOSED_POSITIONS_PATH}') cp
    JOIN esports_markets em USING (market_id)
    WHERE em.game IS NOT NULL
    GROUP BY em.game
    ORDER BY gross_usd_volume DESC
    ) TO '{cache_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    t0 = time.time(); con.execute(sql); print(f"built {cache_path.name} in {time.time() - t0:.1f}s")
    return pd.read_parquet(cache_path)

closed = closed_position_summary(use_cache=True)
display(closed)
bar_chart(closed, "game", "gross_usd_volume", "Closed-position gross volume by game", color="#66A61E")


## 7. Candidate Grid

A practical first-pass grid for latency-arb research is:

- Game: LoL, Dota 2, CS2, Valorant.
- Market type: `live_total_kills`, `objective_or_multikill_prop`, `game_winner`, then `series_moneyline_or_other` as a control.
- Activity gate: historical notional and next-fill availability.
- External source gate: measured source timestamp must beat public stream / CLOB reaction by enough to clear fees, spread, and failed-fill risk.

The table below ranks historical game x market-type cells by a rough score. Treat it as a prioritisation list, not an edge estimate.

In [ ]:

cell = type_hist.merge(lat[["game", "pct_gap_le_30s", "gap_p50_s", "gap_p90_s"]], on="game", how="left")
cell["score"] = (cell["usd_notional"].clip(lower=1).pipe(lambda s: s / s.max()) * 0.65 + cell["pct_gap_le_30s"].fillna(0) * 0.35)
interesting_types = ["live_total_kills", "objective_or_multikill_prop", "game_winner", "series_moneyline_or_other", "first_blood"]
candidate_grid = (cell[cell["market_type_proxy"].isin(interesting_types)]
    .sort_values("score", ascending=False)
    .loc[:, ["game", "market_type_proxy", "fills", "markets_traded", "usd_notional", "pct_gap_le_30s", "gap_p50_s", "gap_p90_s", "score"]]
    .head(30))
display(candidate_grid)
heatmap(candidate_grid, "game", "market_type_proxy", "score", "Candidate grid score: notional + 30s next-fill availability")
